# DX703 Final Project — Final Code Notebook (Part A)

**Due Date:** Sunday, December 6, 2026 at 11:59 p.m. (2-hour, 1-minute grace period)

**No late submissions!**

**Submit:** This completed `.ipynb` notebook and the separate technical report (Part B, `.pdf`).  

## Purpose

Build on your work from the **DX703 Final Project Milestone: Data Exploration and Baseline Modeling**. Using your previously selected dataset and established train/validation/test split, you will:

- Develop and evaluate a custom neural network, including experiments to improve its performance.
- Implement a pretrained model and investigate transfer-learning strategies.
- Compare both approaches with your milestone baseline using validation results.
- Select and finalize the best model based on predictive performance and computational constraints.
- Evaluate the selected model on the held-out test set and analyze its errors.

This assignment supports **both Food-101 (image classification) and HuffPost News Category (text classification)**. Use architectures and preprocessing appropriate to your dataset while addressing the same multiclass-classification objectives.

## Grading Overview

| Problem | Focus | Points |
|---|---|---:|
| 1 | Setup, preprocessing, and baseline recap | **10** |
| 2 | Custom neural network and experiments | **35** |
| 3 | Pretrained model and transfer learning | **35** |
| 4 | Comparison, selection, and final training | **30** |
| 5 | Final test evaluation and error analysis | **25** |
| **Total** | | **135** |

## Important Submission Requirements

- **Preserve the notebook structure.** Complete the provided code and Markdown cells without deleting or replacing them. You may add additional cells as needed. Keep all required results and written responses in their designated locations to support reliable grading without rerunning expensive experiments.

- **Submit a notebook with executed results.** Include the relevant outputs, training histories, comparison tables, metrics, and plots. Code alone is not sufficient evidence for full credit if its required results are missing.

- **Complete the written responses.** Answer questions in the designated Markdown cells, replacing the *Your Answer* placeholders. Provide concise but substantive explanations supported by your experimental results. More extensive discussion belongs in the technical report.

- **Document your experiments.** Use organized, reproducible code, consistent random seeds where practical, and clearly identified model configurations. Avoid hard-coded local paths or dependencies on inaccessible private files.

- **Manage computational constraints responsibly.** You may use documented, reproducible subsets when full-dataset experiments are impractical. Preserve the original train/validation/test separation, explain how subsets were selected, and make model comparisons on consistent or explicitly comparable validation data. Do not present subset results as full-dataset results.

- **Protect the held-out test set.** Use validation results exclusively for model development, tuning, and selection. The final model and evaluation protocol must be fixed in Problem 4 before examining test results in Problem 5. Do not revise the model based on test performance.

- **Preserve expensive experimental results.** You are not required to rerun computationally expensive training during grading. Save the relevant outputs, training configurations, and checkpoints when practical. Your notebook should document how the results were obtained and how the final evaluation could be reproduced.

- **Coordinate with the technical report.** The notebook should demonstrate your implementation, experimental evidence, and concise interpretations. The separate technical report (Part B) will present a more complete discussion of methodology, results, limitations, and conclusions.

## Problem 0 — Dataset Selection (0 pts)

Identify the dataset used for your final project. It must be the same dataset you used in the milestone.

**Selected dataset:** Food-101

*Delete the dataset you are not using.*

## Problem 1 — Setup, Preprocessing, and Baseline Recap (10 pts)

Reuse the dataset, split, and baseline developed in your milestone. You may improve your preprocessing pipeline, but document changes and retain a fair validation comparison. **Do not repeat all the milestone EDA or write another lengthy dataset description.**

- **A. Reusable data pipeline (4 pts)**
- **B. Baseline evidence (4 pts)**
- **C. Milestone-to-final transition (2 pts)**

### Part A — Reusable Data Pipeline (4 pts)

**Implementation (3 pts):**

- Set random seeds;
- import required packages;
- load the chosen dataset or saved splits;
- reuse exactly the established train/validation/test partition;
- implement the final preprocessing/data-loading functions or pipelines needed for modeling.
- Fit learned preprocessing only on training data, and apply training-only augmentation only to training examples.
- **Category consolidation (optional):** You may combine closely related target categories when justified by their meaning (e.g., *Arts* and *Arts and Leisure* in HuffPost). If you consolidated categories in the milestone, preserve that mapping. If you decide to consolidate categories now, apply the mapping consistently across all three splits without changing which observations belong to each split. **Category consolidation is not required.**
- Show split sizes and relevant shape/type sanity checks. **Do not evaluate on the test set.**

**Explanation (1 pt):** Summarize any preprocessing changes relative to the milestone and why they were made. If you consolidated target categories, explain which categories were combined and why. If you changed the category definitions since the milestone, explain how this affects comparisons with your original baseline; models using different category definitions are not directly comparable. If there are no changes, say so.

In [4]:
# P1.A — Load your established splits and construct/reuse preprocessing.
# Add imports, reproducibility settings, dataset-loading code, and pipeline code here.
# You may insert further code cells after this one.

# ============================================
# Useful Imports
# ============================================

# --- Standard Libraries
import time
import os
import math
import random
from collections import Counter
import warnings
import shutil
from pathlib import Path

# Suppress Warnings
warnings.filterwarnings('ignore', category=UserWarning, module='PIL.TiffImagePlugin')
os.environ["PYTHONWARNINGS"] = "ignore"

# --- Core Data / Numerics
import numpy as np
import pandas as pd

# --- Visualization
import matplotlib.pyplot as plt
import seaborn as sns              # optional
import matplotlib.ticker as mticker  # optional (for formatted axes)

# --- Progress Tracking
from tqdm import tqdm                # optional (nice for loops)

from IPython.display import display

# --- TensorFlow / Keras (Deep Learning)
import tensorflow as tf
from tensorflow.keras import layers, models, Input, callbacks, regularizers, initializers
from tensorflow.keras.callbacks import Callback, EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam, AdamW
from tensorflow.keras.optimizers.schedules import CosineDecay, ExponentialDecay
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array
from tensorflow.keras.layers import (
    Dense, Dropout, Flatten, MaxPooling2D, Conv2D,
    SeparableConv2D, GlobalAveragePooling2D, GlobalMaxPooling2D, BatchNormalization
)

# --- (Optional) Classical ML Baseline Tools
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# --- Hugging Face Datasets
from datasets import load_dataset, load_from_disk, DatasetDict
from datasets.features import ClassLabel

# --- Image Tools
from PIL import Image as PILImage
from PIL import Image, ImageStat, ImageOps
from datasets import Image as HFImage
import io



# ============================================
# Global Configuration & Small Utilities
# ============================================

# Reproducibility
random_seed = 42
random.seed(random_seed)
np.random.seed(random_seed)
tf.keras.utils.set_random_seed(random_seed)         # sets Python, NumPy, and TensorFlow seeds
tf.config.experimental.enable_op_determinism()      # controls hardware execution order for reproducible floating points

# change image size if needed
IMAGE_SIZE = (512, 512)

# =============================
# IMAGES TO DROP, CROP, RELABEL
# =============================

# image index with associated coordinates for crop box
crop_imgs = {
    50524: (160, 0, 380, 260),       # image collage cropped to just the pho
    50612: (300, 240, 512, 512),     # cropping image of pho just to the bowl of pho rather than the whole thing since theres also a sandwich
    33410: (60, 220, 180, 512),      # cropping image to strawberry shortcake and leaving out the cup cakes to not confuse the model between the two labels 
    74241: (0, 310, 470, 512),       # cropping grilled salmon dish to just the salmon since french fries may confuse the model because it is also a label.
    74210: (80, 200, 420, 320),      # wok of salmon with chef and somewhat busy background. cropping into just the food.
    17799: (120, 260, 220, 350),     # cropping image of woman holding container with hot dog to keep main focus of pixels on hot dog and not other details like people or background details.
    17995: (160, 250, 420, 400),     # cropping image of hot dog at baseball game to get rid of background arena details
    17907: (160, 160, 340, 300),     # cropping to just the hot dog since it is a far away top down view of the table
    17986: (140, 300, 400, 400),     # cropping hot dog on elephant stuffed animal's lap so the focus is just on the hot dog.
    17494: (100, 300, 420, 420),     # cropping image of hot dog at baseball game to get rid of background arena details
    17750: (120, 280, 360, 512),     # cropping image of hot dog at baseball game to get rid of background arena details
    17442: (240, 420, 320, 490),     # cropping image of a woman by a table holding a hot dog to just the hot dogs on the table
    17987: (120, 160, 240, 280),     # cropping image around just the hot dog in the person's hands because the rest of the image is empty space/inside of the car
    71579: (100, 320, 380, 512),     # cropping churros to remove background details of people and the stand
    71564: (260, 240, 380, 380),     # cropping out instagram UI to focus just on churros
    71867: (200, 320, 500, 480),     # cropping out restaurant background to focus on churros
    32368: (100, 190, 400, 340),     # cropping out smartphone UI to focus on crabcake sandwich
    58642: (20, 240, 220, 400),      # cropping out person so just the curry is shown without any side dishes as well
    95957: (280, 100, 480, 420),     # cropping out side dish for filet mignon
    60111: (40, 100, 260, 220),      # cropping out side dishes for filet mignon
    32017: (180, 180, 320, 400),     # cropping out background to focus just on breakfast sandwich
    32204: (100, 240, 420, 500),     # cropping out city background to focus on breakfast sandwich
    31566: (40, 200, 280, 300),      # cropping out chef and kitchen to focus on breakfast sandwich
    39531: (260, 260, 512, 512),     # samosa but cropping out collage of menu/food truck to focus on samosa
    39720: (260, 280, 460, 380),     # samosa but cropping to just the dishes with samosas instead of the whole image with the server and background
    9505: (20, 400, 300, 512),       # guacamole but cropped to just the guacamole the man is making
    9074: (180, 380, 400, 512),      # guacamole but cropped to just the guacamole the man is making
    9198: (180, 0, 460, 240),        # guacamole but cropped to just the bowl of guacamole since it is on the far end of the table
    9120: (260, 280, 512, 512),      # guacamole but cropped to bottom left image in the collage since its the guacamole part of the collage
    9097: (300, 280, 420, 280),      # guacamole but cropping onto bowl of guacamole since there's a lot in the background and on the table
    9113: (340, 200, 500, 360),      # guacamole but cropped to guacamole bowl and remove the bag of chips that takes up more of the image
    70123: (200, 300, 390, 440),     # poutine but cropped to just the poutine and getting rid of graphic design in background
    70274: (200, 380, 400, 512),     # poutine but cropped around food to remove the person
    75290: (260, 260, 500, 500),     # collage of images and cropped down to spaghetti bolognese
    75256: (140, 0, 340, 100),       # crop to the spaghetti bolognese on the far end of table
    22945: (240, 100, 480, 300),     # cropped on the chocolate mousse and remove signs/background items
    26536: (200, 200, 400, 420),     # cropped out baseball game to focus on just ice cream bowl
    26450: (160, 60, 400, 260),      # cropped out background to focus on ice cream cup
    53520: (0, 60, 512, 275),        # cropped out label from container of edamame so the focus is on the food
    53269: (100, 320, 300, 512),     # cropped out other dishes to focus on edamame
    53894: (310, 120, 512, 512),     # cropped out soda can to focus on edamame
    53598: (110, 110, 220, 200),     # cropped to focus on bowl of edamame in the back of the table
    52710: (160, 220, 320, 340),     # cropped out other food in fridge to focus on lobster roll sandwiches
    53120: (180, 140, 420, 360),     # cropped out lobster roll sandwich since it barely shows from the wrapper. Will switch to onion rings label
    52722: (240, 140, 512, 420),     # cropped out chips bag to focus on lobster roll sandwich
    2051: (0, 220, 512, 512),        # cropped out woman to focus just on ramen
    1810: (120, 80, 512, 280),       # crop out most of the advertisement sign to focus on the ramen
    13009: (140, 280, 400, 420),     # cropped out side dishes to focus on fried rice
    12887: (300, 160, 500, 320),     # collage of photos, cropped to fried rice image
    8053: (230, 280, 360, 410),      # cropped to pizza and crop out baby/background
    8079: (60, 260, 440, 420),       # cropped to pizza on the stands
    620: (160, 120, 300, 260),       # beignet but cropped to just the food and leave out person/background
    44982: (60, 300, 360, 512),      # lobster bisque but cropped to just the food and leave out the person/background
    46938: (160, 280, 360, 440),     # cropping to french onion soup and remove person/background
    57197: (260, 20, 490, 240),      # collage of nachos but cropping to just one image of nachos
    57706: (20, 20, 490, 260),       # ad for nachos but cropping to just upper nachos image
    61278: (0, 320, 512, 512),       # caprese salad but cropping out background that takes up most of the image.
    60773: (0, 280, 460, 512),       # caprese salad but cropping out background hills and pool to focus on food
    2626: (140, 260, 280, 380),      # hamburger but cropping out background/mountains that take up a lot of space
    2659: (0, 260, 512, 512),        # hamburger but cropping out upper half of collage that shows a building
    4893: (60, 320, 440, 512),       # pork chop but cropping out woman to focus just on food
    66761: (0, 320, 300, 512),       # spaghetti carbonara but cropping out background to focus on food on bottom left corner
    66849: (260, 20, 480, 250),      # cropping 4 image collage to just the spaghetti carbonara
    59958: (0, 260, 340, 440),       # omelette but cropping out energy drink
    20040: (60, 200, 450, 450),      # fish and chips collage but cropping to main image of fried fish on top of fries
    19804: (60, 280, 512, 512),      # fish and chips but cropping out girl
    10849: (0, 300, 140, 512),       # miso soup but cropping out child to focus on bowl of soup
    865: (0, 380, 512, 512),         # prime ribs but cropping out background since it takes up most of the image
    1247: (70, 320, 440, 512),       # prime ribs but cropping out orange item in background that takes up most of the image
    770: (0, 340, 512, 512),         # prime ribs but cropping out person that takes up more than half the image
    18093: (0, 0, 512, 200),         # steak but cropping out bottom half of wine bottles
    18665: (120, 100, 390, 370),     # steak but cropping out instagram UI
    14852: (140, 320, 320, 512),     # hummus but cropping out people/background to focus on hummus dish
    40728: (140, 220, 380, 460),     # french fries but cropping out tree that takes up most of the image
    41071: (220, 320, 420, 460),     # french fries but cropping out blurry people on sides
    6342: (60, 240, 320, 512),       # sashimi but cropping out person that takes up most of the image
    5367: (330, 0, 512, 512),        # greek salad but cropping out sign
    5772: (30, 30, 480, 480),        # greek salad but cropping out frame
    20878: (0, 280, 240, 380),       # caesar salad but cropped out server/background details
    20863: (260, 320, 512, 512),     # caesar salad but crop out labeling that takes up half of image
    65990: (340, 20, 490, 240),      # baby back ribs but crop out other collage images
    65514: (140, 220, 420, 480),     # baby back ribs but crop out graphic design/packaging
    28829: (120, 380, 380, 460),     # foie gras but cropping out chef/background details
    56633: (240, 160, 480, 360),     # peking duck but cropping to dish of dumplings to relabel it since duck dish is ambiguous
    56949: (60, 200, 240, 440),      # peking duck but cropping to dish to remove background that takes up more space
    28206: (160, 340, 280, 480),     # tacos but cropping out child
    34613: (100, 380, 340, 512),     # mussels but cropping out woman
    34653: (180, 140, 340, 260),     # mussels but cropping out menu
    34877: (180, 180, 400, 330),     # mussels but cropping out sign
    96260: (260, 280, 480, 480),     # macarons but cropping out background items
    61952: (80, 50, 240, 490),       # macarons but cropping out other two collage images to focus on one
    61622: (0, 400, 512, 512),       # macarons but cropping out half of plastic container
    61667: (100, 80, 240, 512),      # macarons but cropping out to focus just one food and not signs
    61865: (180, 0, 500, 140),       # macarons but cropping out black and white graphic design that covers most of the image
    41966: (0, 380, 360, 512),       # chicken quesadilla but cropping out background that takes up most of the image
    41283: (0, 200, 300, 380),       # chicken quesadilla but focus on food
    41911: (10, 260, 490, 490),      # chicken quesadilla but cropped out top image in collage
    41869: (0, 260, 512, 470),       # chicken quesadilla but cropped out bag/table to focus on food
    37310: (0, 300, 512, 512),       # falafel but cropping out background details that cover half the image
    37220: (260, 0, 512, 512),       # falafel but cropping out left image in 2 image collage
    54136: (100, 240, 220, 400),     # pulled pork sandwich but cropping out person/background
    54273: (60, 300, 300, 512),      # pulled pork sandwich but crops out background
    24409: (20, 260, 260, 490),      # pad thai but crop out other food/drinks in collage
    52271: (0, 0, 512, 240),         # takoyaki but cropped on the pan
    51954: (60, 320, 340, 440),      # takoyaki but cropped to focus on food not the chefs
    52220: (0, 0, 512, 200),         # takoyaki but cropped to focus on it instead of octopus
    52084: (0, 140, 270, 440),       # takoyaki but focus on main group of them
    52066: (260, 60, 512, 440),      # takoyaki but crop out chef/grills
    52412: (180, 220, 320, 400),     # takoyaki but crop out person
    14149: (120, 280, 320, 512),     # panna cotta but crop out kitchen
    4169: (60, 160, 240, 340),       # chicken wings but cropping out waffles since waffles are also a label
    23639: (16, 300, 420, 480),      # lasagna but cropped out person/background to focus on plate
    23702: (40, 290, 320, 460),      # lasagna but cropped out window blinds/wine bottle
    11874: (200, 110, 512, 430),     # apple pie but crop out coffee and background details
    11486: (200, 200, 420, 380),     # apple pie but cropped out pie since plastic hides it. Recropped to burger to relabel as hamburger
    11512: (0, 200, 512, 512),       # apple pie but crop out background/sun glare
    30317: (0, 260, 512, 512),       # frozen yogurt but crop out restaurant photo in the 2 image collage
    30656: (0, 0, 280, 512),         # frozen yogurt but crop out dog on sidewalk
    73384: (0, 280, 300, 440),       # mac and cheese but crop out person/background
    73273: (120, 300, 420, 512),     # mac and cheese but cropping out background
    73134: (180, 280, 380, 440),     # mac and cheese but crop out person and most of the garlic bread
    46099: (260, 360, 500, 500),     # waffles but cropped to waffles from the 3 image collage
    31452: (80, 0, 440, 260),        # seaweed salad but cropping out labeling/plastic
    68269: (240, 0, 512, 512),       # bibimbap but cropping out food truck from photo collage
    68257: (20, 60, 240, 320),       # bibimbap but cropping out second dish in 2 photo collage
    68342: (60, 20, 320, 320),       # bibimbap but crop out white space/drinks
    3349: (20, 120, 512, 220),       # bruschetta but cropped to board of bruschetta in the back
    72153: (60, 310, 420, 460),      # cannoli but cropped out sparkler and person
    62743: (20, 270, 170, 400),      # croque madame but cropping out fries/background to not confuse model since fries is also a label
    62460: (100, 280, 300, 440),     # croque madame but cropping out drink/table 
    25443: (60, 280, 160, 512),      # cheesecake but focusing on plain cheesecake in left corner
    25109: (260, 180, 500, 500),     # cheesecake but 3 image collage of different desserts so its cropped around the cheesecake only
    66454: (40, 190, 380, 300),      # ravioli but it looks like mussels and some sort of sauce so cropping around mussels to relabel image as mussels
    35677: (280, 100, 420, 320),     # dumplings but cropping out kitchen equipment
    48440: (220, 0, 360, 180),       # clam chowder but cropping out person/background
    48581: (0, 120, 220, 220),       # clam chowder but cropping out sandwich/background
    79142: (220, 160, 512, 360),     # chocolate cake but cropping out tiramisu
    22074: (220, 310, 340, 370),     # pancakes but cropping out child and dishes of different types of food
    21932: (40, 140, 140, 250),      # pancakes but cropping out pancake maker machine/bowl/syrup
    81496: (100, 320, 280, 460),     # garlic bread but cropping out cat
    17166: (180, 120, 480, 300),     # garlic bread but cropping out stuffed animal/background
    17003: (100, 260, 360, 500),     # garlic bread but cropping out menu and container
    16963: (60, 100, 360, 300),      # garlic bread but cropping out person
    69006: (200, 180, 460, 460),     # cup cakes but cropping out sign/frame
    44152: (20, 100, 210, 400),      # french toast but cropping out chips/table
    74466: (280, 80, 420, 420),      # paella but cropping out chef/background
    43288: (140, 280, 380, 360),     # huevos rancheros but cropping out woman
    43441: (0, 190, 420, 512),       # huevos rancheros but cropping out beach background
    67514: (220, 220, 440, 380),     # sushi but cropping out other dishes
    8888: (60, 140, 460, 400),       # eggs benedict but cropping out frame
    3894: (80, 200, 440, 400),       # chicken wings but cropping on fries because it takes up more of the focus. relabeling to french_fries
    4171: (160, 180, 330, 320),      # chicken wings but 3 image collage so I cropped to the wings
    24746: (0, 170, 240, 340),       # pad thai but 2 image collage so I cropped to one of the pad thai images
    18115: (120, 80, 420, 220),      # steak but cropping around fries since they are more visible. relabeling to french_fries
}

# image indices to drop (non-relevant images, ambiguous food types, extremely poor quality images)
IMAGES_TO_DROP = [
    33022,  # strawberry_shortcake image that is some kind of fruit drink
    33638,  # strawberry_shortcake image that could be a strawberry shortcake but is very ambiguous and not clear
    33713,  # strawberry_shortcake image that could be a strawberry shortcake but is very ambiguous and not clear
    33317,  # strawberry_shortcake image but it is ambiguous since there are no visible strawberries and the cake is covered in almond slices
    33007,  # strawberry_shortcake image but it is ambiguous since it is a green cake with no visible strawberries
    33189,  # strawberry_shortcake image but it is a cup of ice cream or slushy with strawberry looking flavors
    33444,  # strawberry_shortcake image but there are cookies that are strawberry and cream flavor from the package in the background
    33182,  # strawberry_shortcake image but it is collage of different cakes which makes it more ambiguous. Also no strawberries visible.
    29672,  # grilled_cheese_sandwich image but it is unclear what they are. Massive pile of sandwiches with egg maybe?
    29486,  # grilled_cheese_sandwich image but it wrapped in paper so the model wouldn't know if it is a grilled cheese sandwich.
    29908,  # grilled_cheese_sandwich image but it is ambiguous since it could be a burger or a grilled cheese on a burger bun. Dropping rather than relabeling to hamburger.
    7190,   # risotto image but it is ambiguous. Dark food with no clear indication of what it is.
    6994,   # risotto image but dropping because main focus is on a man with a dog. There is a table of ingredients but it will confuse the model.
    6941,   # risotto image but it is ambiguous. Not sure what kind of food it is.
    37614,  # scallops image but it is a night outside resort location with no food.
    37711,  # scallops image but it is ambiguous because a chef is cooking food on a grill and the food is small compared to the majority of the image.
    50931,  # pho image but it is not food at all. purple background with designs.
    2414,   # hamburger image of outside a restaurant with a man holding a hamburger. Dropping because it is more ambiguous while the goal is to detect real food.
    15306,  # carrot_cake image but it is a lot of small servings with very dark lighting and people. Dropping because it might cause issues with model.
    26400,  # ice_cream image of cartons inside a fridge. Dropping because it is the container not the actual food.
    50380,  # pho image but dropping because of computer screen and mouse icon. Its more of an ambiguous image.
    92702,  # pho image but it looks like a kind of cake. Unsure what kind so I will drop it.
    50812,  # pho image but dropping because poor quality containers blur whats inside and it is a big difficult to tell what it is.
    50979,  # pho image but dropping because it is just a leaf on a plate.
    50652,  # pho image but it is actually an empty cup.
    50470,  # pho image but it is actually a bridge at sunset.
    50883,  # pho image but it is most likely tempura vegetables and other dishes.
    74120,  # grilled salmon image but it is a menu.
    73632,  # grilled salmon but ambiguous image since it is covered in sauce to the point you cant tell what it is.
    73699,  # grilled salmon salad but you can barely see the salmon.
    74137,  # grilled salmon but dropping because it doesnt look like salmon and is extremely blurry.
    73735,  # grilled salmon but dropping because it is too ambiguous with several dishes and a person. Not worth cropping.
    74046,  # grilled salmon but doesnt look like it. unsure what kind of food it is.
    17695,  # hot dog but it is actually Thomas the Tank Engine
    17905,  # hot dog but dropping because it is just a hot dog restaurant with no visible food
    17633,  # hot dog but dropping because it looks more like bread (baguette or some kind of long loaf)
    17556,  # hot dog but dropping because it looks like something fried, unsure what it is and poor quality image due to filters/text/etc.
    17786,  # hot dog but dropping because it looks like a food truck with no visible hot dogs
    17345,  # hot dog but dropping because it is a food stand with no visible hot dogs.
    17323,  # hot dog but dropping because it looks like a food truck with no visible hot dogs
    17481,  # hot dog buns but no hot dogs
    17864,  # hot dog but the image is ambiguous. it looks like there might be a hot dog in the wrap but cropping would be too small
    17723,  # hot dog but dropping because it is a corn dog
    17463,  # hot dog but dropping because it is a corn dog
    17453,  # hot dog but it is actually just a photo of a person
    71522,  # churros but it is actually a photo of a man
    71764,  # churros but it is a sign with no food
    71751,  # churros but it is ambiguous. Not ice cream but some kind of sweets
    71325,  # churros stand but churros is too small to crop on
    71996,  # churros but dropping because it is ambiguous. could be sprinkles/chocolate on churros or it could be a donut
    58999,  # chicken curry but dropping because it is just the packaging for butter chicken curry
    58991,  # chicken curry but dropping because it is a baby
    58673,  # chicken curry but dropping because it's two people
    58543,  # chicken curry but dropping because it is just a menu
    58947,  # chicken curry but dropping because it is just naan with a little bit of curry sauce on the bowl.
    58725,  # chicken curry but dropping because it is just people eating on a plane
    59229,  # chicken curry but dropping because it is looks like vegetables and rice
    58935,  # chicken curry but dropping because it is a table of food with low quality. Unsure what food there is.
    59174,  # chicken curry but dropping because it is ambiguous. Many dishes at a kitchen to serve
    60457,  # filet mignon but dropping because it is a group of people
    60706,  # filet mignon but dropping because it is a group of people
    60223,  # filet mignon but dropping because it is a bit ambiguous. definitely meat but low quality
    60334,  # filet mignon but dropping because it is a bit ambiguous. definitely meat but covered mostly in cheese
    95978,  # filet mignon but dropping because it doesnt look like it is filet mignon.
    60392,  # filet mignon but dropping because it is ambiguous. Wrapped in bacon and sauce, unsure what it is.
    32168,  # breakfast sandwich but dropping because it is just a drink and napkins
    31820,  # breakfast sandwich but dropping because it is some sort of microwave meal
    32024,  # breakfast sandwich but dropping because it is some sort of microwave meal
    31804,  # breakfast sandwich but dropping because it is just 3 drinks
    32167,  # breakfast sandwich but dropping because it is wrapped up and cant see the actual sandwich
    32117,  # breakfast sandwich but dropping because crop would require too small image and lose a lot of detail
    31567,  # breakfast sandwich but dropping because it is ambiguous. Ingredients are being made but its not all combined
    32090,  # breakfast sandwich but dropping because it is just a black square
    39167,  # samosa but dropping because it is a person
    39329,  # samosa but dropping because it is a food stand with no visible samosas big enough to crop to
    39566,  # samosa but dropping because I am not sure what food it is
    9529,   # guacamole but dropping because it is a person
    9079,   # guacamole but dropping because it is mainly graphics and the container
    9073,   # guacamole but dropping because it is the ingredients just not made yet
    9578,   # guacamole but dropping because it is too small to crop in on fresh made guacamole in the restaurant
    9155,   # guacamole but dropping because it is unclear if there is any guacamole at the table
    78871,  # guacamole but dropping because it is a bit ambiguous. There is some guacamole in a wrap and a side of chips. 
    9309,   # guacamole but dropping becauae it is an empty bowl
    9671,   # guacamole but dropping because it is just tortilla chips
    69781,  # poutine but dropping because it is a poutine food truck not the food
    75474,  # spaghetti bolognese but dropping because it is an office space, not food.
    100820, # spaghetti bolognese but dropping because it is ambiguous. It looks like only sauce in a container
    75024,  # spaghetti bolognese but dropping because it is clearly not the dish. Looks like a brown meat sauce.
    75547,  # spaghetti bolognese but dropping because it doesnt look like it. Gormet dish of a different pasta with some tomato cubes
    63849,  # gyoza but dropping because it is ambiguous what the chef is making
    97109,  # gyoza but dropping because it is unclear what it is. gormet kind of salad?
    23060,  # chocolate mousse but dropping because it is unclear based on lighting/quality. 
    22971,  # chocolate mousse but dropping because it might confuse model. edible gold leaf over most of the chocolate mousse
    23191,  # chocolate mousse but dropping because it is poor quality and hard to tell if it is cup cakes or mousse
    22531,  # chocolate mousse but dropping because it looks more like chocolate candy
    22838,  # chocolate mousse but dropping because it is the wrapper of chocolate mousse candy
    26346,  # ice cream but dropping because it is the flavor menu board
    26654,  # ice cream but dropping because they are cartons
    26493,  # ice cream but dropping because it is an ice cream stand with only cartons visible
    26321,  # ice cream but dropping because it is a carton
    26833,  # ice cream but dropping because it is a freezer with ice cream. no ice cream visible
    26384,  # ice cream but dropping because it only shows a chef
    26529,  # ice cream but dropping because it is the outside of an ice cream shop
    26656,  # ice cream but dropping because it is an ice cream stand with not enough to crop to
    26675,  # ice cream but dropping because they are cartons
    26345,  # ice cream but dropping because no ice cream visible, only gummy bears and cone
    26839,  # ice cream but dropping because it is an ice cream wrapper
    26820,  # ice cream but dropping because it is an ice cream wrapper
    53918,  # edamame but not able to crop enough of the actual food from the container
    53959,  # edamame but dropping because it looks like some kind of sandwich (no edamame)
    53060,  # lobster roll sandwich but dropping because it is just a food stand
    53237,  # lobster roll sandwich but dropping because it would take too small of a crop to focus on sandwich
    1502,   # ramen but dropping because it is only the ramen cups without showing the food
    2207,   # ramen but dropping because it is a painting of soup
    1958,   # ramen but dropping since you cant see ramen in bowl that the server is holding
    1713,   # ramen but dropping because it is packages of ramen instead of a clearer image of the meal
    2150,   # ramen but dropping because it is a package of ramen instead of a clearer image of the meal
    1952,   # ramen but dropping because it is a package of ramen
    1824,   # ramen but dropping because most of the image is an ad with too small of a crop to focus on ramen
    13444,  # fried rice but dropping because it is a package of sweet apple
    13203,  # fried rice but dropping because it is a man
    13062,  # fried rice but dropping because it is a man
    13079,  # fried rice but dropping because it is a chef
    13152,  # fried rice but dropping because it is vegetables on the grill, not fried rice.
    13395,  # fried rice but dropping because it is vegetables and meat on the grill, not fried rice.
    12780,  # fried rice but dropping because it is unclear what it is. Fried food but not fried rice
    7567,   # pizza but dropping because no pizza shown, only a pizza shop
    8121,   # pizza but dropping because it is a vietnam logo
    8110,   # pizza but dropping because its a pizza box
    8199,   # pizza but dropping because its a kitchen
    7907,   # pizza but dropping because it is a baby
    7915,   # pizza but dropping because its just a restaurant
    8002,   # pizza but dropping because its just restaurant graphic design
    7548,   # pizza but dropping because its just restaurant graphic design
    8195,   # pizza but dropping because its a group of people
    7722,   # pizza but dropping because most of the image are people. They're eating pizza but i think it would confuse the model if I included it
    7888,   # pizza but dropping because its an inside of brick oven 
    7886,   # pizza but its just drinks
    228,    # beignet but its people
    508,    # beignet but its people
    279,    # beignet but its people 
    98,     # beignet but its people
    171,    # beignet but its a sign
    381,    # beignet but its a sign
    165,    # beignet but its a chef and too small to crop on the food
    646,    # beignet but its some other kind of pastry
    355,    # beignet but its a chef
    747,    # beignet but not enough shows to justify cropping (mostly paper bag full of them and 1 small beignet on far left)
    55800,  # bread pudding but mostly container. not worth cropping
    56094,  # bread pudding but its batman
    55583,  # bread pudding but ambiguous because it has ice cream which is also another label
    55700,  # bread pudding but its actually a tomato based dish
    55911,  # bread pudding but its a menu
    44283,  # lobster bisque but its a container
    44401,  # lobster bisque but unsure what food it is. definitely not soup consistency
    44995,  # lobster bisque but its spaghetti and lobster
    44575,  # lobster bisque but its packaging
    44261,  # lobster bisque but its ambiguous. unsure so dropping to be safe
    44590,  # lobster bisque but its a lobster not the soup
    44304,  # lobster bisque but its a lobster not the soup
    44306,  # lobster bisque but its a lobster not the soup
    47013,  # french onion soup but it looks more like pastry
    46557,  # french onion soup but it looks like a different kind of soup
    47022,  # french onion soup but it looks like a different kind of soup
    46796,  # french onion soup but it looks more like pastry
    46861,  # french onion soup but it is ambiguous what type of soup it is
    46667,  # french onion soup but it is ambiguous what type of soup it is
    49980,  # shrimp and grits but its an office
    50151,  # shrimp and grits but its the outside of a restaurant
    49833,  # shrimp and grits but its a water scene
    51683,  # cheese plate but its just a plate no food
    51154,  # cheese plate but its just a plate no food
    57657,  # nachos but its just a child
    57093,  # nachos but its just a person
    60759,  # caprese salad but its just two chefs doing food prep
    2384,   # hamburger but dropping because its not food
    2948,   # hamburger but dropping because its not food
    2677,   # hamburger but dropping because its not food
    2781,   # hamburger but dropping because its not food
    2982,   # hamburger but dropping because its not food
    2573,   # hamburger but dropping because its not food
    2456,   # hamburger but dropping because burgers are completely wrapped
    2510,   # hamburger but its just wrappers and empty plate
    2654,   # hamburger but dropping because its not food
    2912,   # hamburger but its just an empty plate
    70787,  # ceviche but dropping because its ambiguous
    66803,  # spaghetti carbonara but its an egg. Ambiguous that it is deviled or benedict so dropping instead.
    67146,  # spaghetti carbonara but its a different pasta dish (tomato based)
    67324,  # spaghetti carbonara but its a different pasta dish (penne)
    67285,  # spaghetti carbonara but its a completely different dish
    66930,  # spaghetti carbonara but its empty plates
    67065,  # spaghetti carbonara but its a completely different dish
    59852,  # omelette but its a chef and a customer
    95632,  # omelette but its some sort of fried meat
    59637,  # omelette but its boiled eggs
    59863,  # omelette but its a mug
    59408,  # omelette but its a completely different dish
    59381,  # omelette but dropping because its being made by a chef and not finished yet
    15306,  # carrot cake but dropping because its hard to tell what the dishes are (several small servings)
    15340,  # carrot cake but its a restaurant
    15193,  # carrot cake but dropping due to it being wrapped in plastic and hard to crop properly
    20127,  # fish and chips but its a flower
    19842,  # fish and chips but its a drink
    34170,  # beef tartare but its a completely different dish
    33880,  # beef tartare but its an inside of a building
    34313,  # beef tartare but its a chef
    34191,  # beef tartare but its clams. Can't relabel since its not mussels
    33778,  # beef tartare but dropping because there's a bunch of small dishes with dark lighting. not worth cropping
    34128,  # beef tartare but unsure what food it is. covered in cheese
    16461,  # beet salad but its ambiguous. could be strips of beet and other vegetables but also looks like some fries. dropping to be safe
    16364,  # beet salad but its an empty plate
    81100,  # beet salad but ambiguous what kind of salad it is. drop to be safe
    16341,  # beet salad but completely different dish
    15911,  # beet salad but looks like squash or some kind of fruit
    10535,  # miso soup but no visible soup only containers
    10599,  # miso soup but no visible soup only containers
    11201,  # miso soup but no visible soup only containers
    1164,   # prime rib but collage of images and hard to tell if there are ribs in any of them
    1083,   # prime rib but horse logo
    1435,   # prime rib but outdoor seating
    955,    # prime rib but menu
    1181,   # prime rib but menu
    755,    # prime rib but its far in background not worth cropping
    1390,   # prime rib but its far in background not worth cropping
    40224,  # club sandwich but completely different dish
    40476,  # club sandwich but completely different dish
    40008,  # club sandwich but completely different dish
    18735,  # steak but aging meat on racks
    18166,  # steak but building
    18508,  # steak but 2 people
    18467,  # steak but restaurant seating
    63193,  # gnocchi but its people
    63494,  # gnocchi but its people
    63519,  # gnocchi but completely different dish (pasta/vegetables)
    63002,  # gnocchi but completely different dish
    96753,  # gnocchi but completely different dish
    63436,  # gnocchi but completely different dish
    63545,  # gnocchi but completely different dish
    63122,  # gnocchi but completely different dish
    63045,  # gnocchi but completely different dish
    14735,  # hummus but its hummus containers 
    14664,  # hummus but its hummus containers
    14976,  # hummus but its hummus containers
    14497,  # hummus but its a person
    14813,  # hummus but its people
    14264,  # hummus but its a building
    14912,  # hummus but its a different dish
    41052,  # french fries but its McDonalds logo
    40921,  # french fries but its a drink
    40825,  # french fries but its a different dish
    6747,   # sashimi but too much packaging. dropping to not confuse model
    6422,   # sashimi but its ambiguous. menu says sushi but image is zoomed in too much it could be sushi or sashimi
    5282,   # greek salad but its restaurant seating
    5807,   # greek salad but its wrapping
    5424,   # greek salad but its just greens and a lemon slice
    5533,   # greek salad but packing makes it hard to crop well. dropping to not confuse model
    5551,   # greek salad but packing makes it hard to crop well. dropping to not confuse model
    20652,  # caesar salad but doesnt seem like it is
    20556,  # caesar salad but its a person
    20968,  # caesar salad but its a person
    20956,  # caesar salad but its a different dish
    20392,  # caesar salad but its a different dish
    65321,  # baby back ribs but dropping because they are completely wrapped in tin foil
    65535,  # baby back ribs but just bones
    65710,  # baby back ribs but just bones
    28862,  # foie gras but ambiguous what it is
    28977,  # foie gras but ambiguous what it is
    28825,  # foie gras but dropping because its packaging. drop to be safe
    29000,  # foie gras but dropping because its packaging. drop to be safe
    28660,  # foie gras but on grill with large flame. dropping to not confuse model
    28675,  # foie gras but dropping because its packaging. drop to be safe
    28745,  # foie gras but dropping because its packaging. drop to be safe
    29188,  # foie gras but dropping because its packaging. drop to be safe
    28896,  # foie gras but dropping because it would require small crop to focus on it
    56527,  # peking duck but its a chef
    56724,  # peking duck but its a different dish
    56351,  # peking duck but its a different dish
    56840,  # peking duck but its a menu
    56945,  # peking duck but none visible
    56914,  # peking duck but window blinds
    56623,  # peking duck but packaging
    94744,  # peking duck but ambiguous dish
    27823,  # tacos but restaurant signs
    28481,  # tacos but restaurant signs
    27806,  # tacos but restaurant signs
    27842,  # tacos but restaurant signs
    28000,  # tacos but restaurant signs
    27799,  # tacos but swans
    28234,  # tacos but different dish
    28412,  # tacos but different dish
    34666,  # mussels but ambiguous dish
    34750,  # mussels but seafood stand
    87470,  # mussels but clams
    35107,  # mussels but menu
    62077,  # macarons but flowers
    62227,  # macarons but dropping because crop would be too small. actual macarons are quite small compared to the rest of the composition
    62181,  # macarons but ambiguous image
    41783,  # chicken quesadilla but blurry image with main focus on drink. not worth cropping
    41529,  # chicken quesadilla but ambiguous angle which makes it hard to tell what the food is
    41787,  # chicken quesadilla but its salad (not greek, caesar, or seafood so dropping)
    27465,  # red velvet cake but a lot of glare and too small to crop
    27380,  # red velvet cake but main cake is green while the red velvet cake is at the bottom. too difficult to crop
    37485,  # falafel collage but too blurry/ambiguous
    37214,  # falafel collage but too blurry/ambiguous
    36870,  # falafel collage with falafel building and other food
    19469,  # tiramisu but it looks more like cake
    54602,  # pulled pork sandwich but its a restaurant
    42675,  # escargots but ambiguous
    42142,  # escargots but its packaging
    42079,  # escargots but ambiguous
    24116,  # pad thai but not food
    24047,  # pad thai but not cooked yet (shows ingredients in outdoor kitchen)
    24407,  # pad thai but chef/kitchen
    24528,  # pad thai but chef/kitchen
    24417,  # pad thai but ambiguous
    24585,  # pad thai but ambiguous
    24272,  # pad thai but ambiguous
    24328,  # pad thai but ambiguous
    52311,  # takoyaki but card game
    51975,  # takoyaki but restaurant
    51774,  # takoyaki but restaurant
    52473,  # takoyaki but too difficult to crop
    52198,  # takoyaki but ambiguous
    51997,  # takoyaki but food stand rather than the food
    13664,  # panna cotta but its pudding
    4281,   # chicken wings but its CNN news logo
    4330,   # chicken wings but its a stuffed animal
    4108,   # chicken wings but its a group of people
    3904,   # chicken wings but in marinade bag rather than on plates/grills/etc.
    23754,  # lasagna but its broken glass and text
    23643,  # lasagna but collage of photos with food and people
    23367,  # lasagna but its people
    23472,  # lasagna but its woodchips at a playground
    23767,  # lasagna but its a restaurant
    23292,  # lasagna just packaging
    11437,  # apple pie but ambiguous sandwich looking food
    30290,  # frozen yogurt but restaurant
    30058,  # frozen yogurt but restaurant
    30423,  # frozen yogurt but restaurant
    30053,  # frozen yogurt but restaurant
    30259,  # frozen yogurt but restaurant
    30667,  # frozen yogurt but restaurant
    30741,  # frozen yogurt but cartons
    30142,  # frozen yogurt but candy ingredients rather than the frozen yogurt
    30181,  # frozen yogurt but candy ingredients rather than the frozen yogurt
    73341,  # mac and cheese but its a lion
    73065,  # mac and cheese but its fried mozzarella sticks
    46477,  # waffles but its people
    46214,  # waffles but its people
    46378,  # waffles but its people
    45805,  # waffles but its baked goods
    57774,  # donut but its a sign not the actual food
    58156,  # donut but its a sign not the actual food
    58224,  # donut but its a restaurant
    58009,  # donut but its a restaurant
    57927,  # donut but its a restaurant
    58301,  # donut but its shaped like a person and ambiguous. dropping to not confuse model on typical donuts
    68901,  # bibimbap but its a keyboard
    68384,  # bibimbap but its a keyboard
    68869,  # bibimbap but its a restaurant
    76825,  # bruschetta but its pretzel sticks
    3006,   # bruschetta but its a box
    3402,   # bruschetta but its boxes/books
    72217,  # cannoli but ambiguous what kind of dessert it is
    62822,  # croque madame but it looks like salad (plain, unable to relabel)
    62655,  # croque madame but its a different dish
    62656,  # croque madame but it looks like crepes with an egg
    24949,  # cheesecake but its a restaurant
    49370,  # beef carpaccio but it looks like salad with fish (not sure if its seaweed salad so dropping to be safe)
    49299,  # beef carpaccio but thicker pieces of meat in a cup
    92140,  # beef carpaccio but ambiguous 
    66565,  # ravioli but its a food truck
    66250,  # ravioli but its the packaging
    66289,  # ravioli but its people
    66724,  # ravioli but its ambiguous
    35911,  # dumplings but dropping because too many types of food with low quality image not worth cropping
    35899,  # dumplings but its a menu
    35446,  # dumplings but its ambiguous. maybe cupcakes but not sure so I am dropping it.
    48145,  # clam chowder but its fruit
    48678,  # clam chowder but its traffic
    10092,  # chocolate cake but dropping because its design makes it ambiguous (colored in fondant)
    21983,  # pancakes but not pancake batter so dropping to not confuse model
    22267,  # pancakes but ambiguous. maybe red velvet cake, something fried, etc.
    22259,  # pancakes but it looks like crepes
    22213,  # pancakes but its ambiguous
    22308,  # pancakes but its still batter so dropping to be safe
    16805,  # garlic bread but its mostly packaging
    16662,  # garlic bread but ambiguous
    69201,  # cup cakes but dropping because packaging covers most of it
    69475,  # cup cakes but dropping because packaging covers most of it
    44141,  # french toast but its a restaurant
    43638,  # french toast but its ambiguous (could be a salad with garlic bread, unsure)
    43606,  # french toast but its ambiguous (could be pancakes with fruits/syrup, unsure)
    74395,  # paella but its a chef (paella too small to justify cropping)
    74488,  # paella but its a restaurant
    74323,  # paella but ambiguous because its still liquidy and being cooked
    45496,  # onion rings but its a completely different food/wrapper
    38423,  # oysters but wrapped up in netting so dropping to not confuse model
    38542,  # oysters but ambiguous because its fried
    47492,  # spring rolls but its a salad (unsure what type so dropping)
    47779,  # spring rolls but its packaging
    47319,  # spring rolls but its different food entirely
    47613,  # spring rolls but its ambiguous
    43046,  # huevos rancheros but ambiguous. dont see any eggs
    43076,  # huevos rancheros but looks like different food
    43194,  # huevos rancheros but looks like different food
    67861,  # sushi but its a restaurant sign
    67565,  # sushi but its a restaurant sign
    68081,  # sushi but its a restaurant
    36686,  # creme brulee but its a restaurant
    36686,  # creme brulee but its a chef
    36657,  # creme brulee but its a chef
    78569,  # eggs benedict but its different food
    8516,   # eggs benedict but its different food
    61951,  # macarons but too blurry to tell
    17482,  # hot dog but different kind of sandwich
    17665,  # hot dog but different kind of sandwich
    4245,   # chicken wings but copy of image 3894
    46123,  # waffles but dropping because its mostly whipped cream which might confuse model to classify it as ice cream
    91235,  # waffles but dropping because its mostly whipped cream which might confuse model to classify it as ice cream
    22486,  # pancakes but ambiguous
    29883,  # grilled cheese sandwich but its a sub
    50878,  # pho but its ambiguous. could be ramen but dropping to be safe
    57379,  # nachos but looks like plain tortilla chips
]

# image indices with image label fixes - found mislabeled images from EDA notebook
IMAGES_TO_RELABEL = {
    33020: 'ice_cream',             # strawberry_shortcake but it is a refrigerator of ice cream flavors
    33112: 'pancakes',              # strawberry_shortcake but it is a plate of pancakes with strawberries
    33305: 'cup_cakes',             # strawberry_shortcake but it is 2 cupcakes and maybe one is strawberry shortcake flavor but the other is a chocolate flavor
    33295: 'chocolate_cake',        # strawberry_shortcake but it is a chocolate cake
    33179: 'cup_cakes',             # strawberry_shortcake because it is a strawberry shortcake flavored cupcake but it might be a bit misleading for the model so I'll consider it cup_cakes
    33021: 'cup_cakes',             # strawberry_shortcake because it is a strawberry shortcake flavored cupcake but it might be a bit misleading for the model so I'll consider it cup_cakes
    6888: 'mussels',                # risotto but its more ambiguous. The mussels stand out the most in the image so I'll relabel it.
    50496: 'spring_rolls',          # pho but the main food are likely spring rolls.
    50525: 'spring_rolls',          # pho but it is most likely Vietnamese fried spring rolls over rice noodles and vegetables.
    50826: 'beef_carpaccio',        # pho image but it is thin slices of raw meat. Most likely beef carpaccio since they are very thin slices of meat
    50362: 'beef_carpaccio',        # pho image but it is thin slices of raw meat. Most likely beef carpaccio since they are very thin slices of meat
    73841: 'onion_rings',           # grilled salmon but a large portion of the image are onion rings, so I will relabel it to that.
    9334: 'nachos',                 # guacamole but it is a lot of tortilla chips with guacamole, salsa, and cheese
    22931: 'chocolate_cake',        # chocolate mousse but it looks more like chocolate cake
    22520: 'chocolate_cake',        # chocolate mousse but it looks more like chocolate cake
    53507: 'fried_rice',            # edamame but it is actually fried rice
    53717: 'sashimi',               # edamame but it looks like raw seafood like sashimi
    52980: 'sushi',                 # lobster roll sandwich but it looks like sushi
    53120: 'onion_rings',           # lobster roll but barely shows so cropped image to onion rings and relabel.
    7921: 'french_fries',           # pizza but its actually a cup of french fries
    55869: 'chocolate_cake',        # bread pudding but it looks like chocolate cake or mousse. thicker than mousse though
    71069: 'tacos',                 # ceviche but its actually tacos
    33966: 'shrimp_and_grits',      # beef tartare but its actually shrimp and grits
    56633: 'dumplings',             # peking duck but ambiguous so cropped to dumplings. relabeled to dumplings
    27487: 'macarons',              # red velvet cake but its actually macarons
    19439: 'ice_cream',             # tiramisu but its actually tiramisu ice cream
    11486: 'hamburger',             # apple pie but hamburger is more visible so I cropped onto the burger instead
    58130: 'cup_cakes',             # donut but they are cupcakes
    68568: 'steak',                 # bibimbap but it looks like strips of steak
    72335: 'chocolate_cake',        # cannoli but it is a chocolate cake.
    72549: 'tiramisu',              # cannoli but its tiramisu
    66454: 'mussels',               # ravioli but its a plate with mussels. cropped to focus on the mussels
    22439: 'waffles',               # pancakes but its waffles
    16632: 'strawberry_shortcake',  # garlic bread but it looks like a strawberry shortcake flavored cake
    36249: 'chocolate_cake',        # creme brulee but its chocolate cake
    3894: 'french_fries',           # chicken wings but cropped around fries since it took up more of the image. relabeling to french fries
    18115: 'french_fries',          # steak but fries are more visible so I cropped on the fries and relabeled to french fries
}

# image indices of duplicate and near duplicates to drop. Near duplicates are the same images just with an effect applied (added blur, grain, text, etc.) - found through cleanvision Imagelab in EDA notebook
DUPLICATES_TO_DROP = [
    
    # EXACT DUPLICATES
    636,    # beignets
    1334,   # prime_rib
    17999,  # steak
    5680,   # greek_salad
    9791,   # chocolate_cake
    9817,   # chocolate_cake
    79522,  # fried_calamari
    13395,  # fried_rice
    80044,  # panna_cotta
    79965,  # panna_cotta
    14700,  # hummus
    80776,  # beet_salad
    17057,  # garlic_bread
    20744,  # caesar_salad
    82751,  # pancakes
    23235,  # lasagna
    23241,  # lasagna
    83102,  # lasagna
    83484,  # pad_thai
    28676,  # foie_gras
    31769,  # breakfast_burrito
    32020,  # breakfast_burrito
    86468,  # strawberry_shortcake
    35706,  # dumplings
    36849,  # falafel
    41704,  # chicken_quesadilla
    45282,  # onion_rings
    46935,  # french_onion_soup
    91021,  # french_onion_soup
    91353,  # clam_chowder
    91604,  # beef_carpaccio
    50626,  # pho
    50685,  # pho
    93179,  # edamame
    54743,  # deviled_eggs
    55045,  # deviled_eggs
    57054,  # nachos
    95775,  # caprese_salad
    95894,  # macarons
    63143,  # gnocchi
    65484,  # baby_back_ribs
    97079,  # baby_back_ribs
    65397,  # baby_back_ribs
    97585,  # spaghetti_carbonara
    97804,  # spaghetti_carbonara
    97791,  # spaghetti_carbonara
    69025,  # cup_cakes
    99228,  # churros
    99969,  # grilled_salmon
    73544,  # grilled_salmon
    100151, # paella
    79316,  # apple_pie
    86415,  # strawberry_shortcake
    86785,  # beef_tartare
    100427, # spaghetti_bolognese

    # NEAR DUPLICATES (DUPLICATES BUT VARIATION IN QUALITY - ADDED BLUR, GRAIN/NOISE, TEXT, ETC.)
    453,    # beignets
    757,    # prime_rib
    75622,  # prime_rib
    1366,   # prime_rib
    1391,   # prime_rib
    1364,   # prime_rib
    75636,  # prime_rib
    7976,   # pizza
    11662,  # apple_pie
    12612,  # fried_calamari
    79511,  # fried_calamari
    16070,  # beet_salad
    81199,  # hot_dog
    27358,  # red_velvet_cake
    28717,  # foie_gras
    32710,  # crab_cakes
    86384,  # strawberry_shortcake
    35206,  # dumplings
    37227,  # falafel
    89410,  # escargots
    42784,  # huevos_rancheros
    46668,  # french_onion_soup
    49700,  # shrimp_and_grits
    92398,  # cheese_plate
    52726,  # lobster_roll_sandwich
    52630,  # lobster_roll_sandwich
    92951,  # lobster_roll_sandwich
    53198,  # edamame
    54265,  # pulled_pork_sandwich
    93768,  # deviled_eggs
    57824,  # donuts
    59252,  # omelette
    95101,  # omelette
    63192,  # gnocchi
    97549,  # ravioli
    97458,  # ravioli
    97749,  # spaghetti_carbonara
    98881,  # ceviche
    98843,  # ceviche
    71861,  # cannoli
    72343,  # cannoli
    73721,  # grilled_salmon
    74261,  # paella
    100185  # paella
]

# final image to drop (found after verifying brightness in EDA notebook. must drop in this order for consistent index number)
dark_image_idx = [9021]

# ================
# HELPER FUNCTIONS
# ================

def format_hms(seconds: float) -> str:
    """Convert seconds to HH:MM:SS format."""
    return time.strftime("%H:%M:%S", time.gmtime(seconds))

def normalize_image_size(image, image_size=IMAGE_SIZE):
    """
        Normalizes images to the specified image_size (pixels) while maintaining the original aspect ratio through padding remaining pixels black.
    """
    # handles already loaded PIL Image objects and file paths
    img = image if isinstance(image, Image.Image) else Image.open(image)

    # formats to RGB
    img = img.convert("RGB")

    # Return resized image (padded if scaled up)
    return ImageOps.pad(img, image_size, method=Image.Resampling.LANCZOS, color=(0,0,0))

def resize_batch(batch):
    """
        Processes images in batches for better performance.
    """
    batch['image'] = [normalize_image_size(img, IMAGE_SIZE) for img in batch['image']]
    return batch

def crop_and_normalize(img, idx):
    """
        Checks if image is flagged for cropping. Applies crop and normalizes size. Returned updated image. 
    """
    if idx in crop_imgs:
        crop_box = crop_imgs[idx]

        # crop image
        cropped_img = img['image'].crop(crop_box)

        # rescale to standardized size for model inputs
        resized_img = normalize_image_size(cropped_img, IMAGE_SIZE)

        # update image in the dataset row
        img['image'] = resized_img
    
    return img

def crop_and_normalize(img, idx):
    """
        Checks if image is flagged for cropping. Applies crop and normalizes size. Returned updated image.
    """
    if idx in crop_imgs:
        crop_box = crop_imgs[idx]

        # crop image
        cropped_img = img['image'].crop(crop_box)

        # rescale to standardized size for model inputs
        resized_img = normalize_image_size(cropped_img, IMAGE_SIZE)

        # update image in the dataset row
        img['image'] = resized_img
    
    return img

def drop_image(dataset, indices):
    """
        Filters out images from dataset. Ensures the list of indices are unique.
    """
    drop_set = set(indices)

    filtered_data = dataset.filter(lambda img, idx: idx not in drop_set, with_indices=True)

    if len(drop_set) != len(indices):
        print(f"List of images contained duplicate indices. Dropping {len(drop_set):,} images.")
    else:
        print(f"Dropped {len(drop_set):,} images.")

    print(f"\nNew dataset size: {len(filtered_data):,} images.")
    return filtered_data

def relabel_image(dataset, label_corrections=dict):
    """
        Updates image labels in dataset using a dictionary (index: 'LABEL_NAME')
        Maps the string label to their corresponding numerical ID.
    """
    # list of label names in dataset
    labels = dataset.features['label'].names

    # maps label names to numerical ID label
    label_to_ID = {name: idx for idx, name in enumerate(labels)}

    # inner function to update the image's label based on dictionary key value pair
    def update_label(img, idx):
        if idx in label_corrections:
            new_label = label_corrections[idx]

            # convert string label name to numerical ID
            if new_label in label_to_ID:
                img['label'] = label_to_ID[new_label]
            else:
                raise ValueError(f"Label '{new_label}' at index {idx} not found in dataset features.")
        return img

    # apply relabeling to all images in label_corrections dictionary
    updated_dataset = dataset.map(update_label, with_indices=True)
    print(f"Relabeled {len(label_corrections):,} images.")

    return updated_dataset

def normalize_image_brightness(image):
    """
        Normalizes images brightness by converting image to YCbCr to equalize luminance without affecting color channels.
        After brightness equalization, return the image converted back to RGB.
    """
    # handles already loaded PIL Image objects and file paths
    img = image if isinstance(image, Image.Image) else Image.open(image)

    # ensure image is RGB first (in case any images are still grayscale/other modes)
    img = img.convert("RGB")

    # convert to YCbCr and split channels
    y, cb, cr = img.convert("YCbCr").split()

    # equalize luminance (Y) channel
    y_eq = ImageOps.equalize(y)

    # Return image with equalized brightness converted back to RGB
    return Image.merge("YCbCr", (y_eq, cb, cr)).convert("RGB")


def normalize_brightness_batch(batch):
    """
        Processes image brightness normalization in batches for better performance.
    """
    batch['image'] = [normalize_image_brightness(img) for img in batch['image']]
    return batch

def plot_learning_curves(history, title="Learning Curves"):
    """
        Plots training vs. validation loss and acurracy with the best epoch highlighted
    """
    h = history.history

    # best epoch for minimum validation loss
    best_epoch_idx = int(np.argmin(h['val_loss']))
    best_epoch     = best_epoch_idx + 1
    epochs         = np.arange(1, len(h['loss']) + 1)

    # metrics at best epoch
    best_val_loss  = h['val_loss'][best_epoch_idx]
    best_val_acc   = h['val_accuracy'][best_epoch_idx]
    best_val_f1    = h['val_macro_f1'][best_epoch_idx]

    # vertically stacked subplots sharing x-axis (epochs) to measure loss, accuracy, and macro f1
    fig, axes = plt.subplots(3, 1, figsize=(8, 7), sharex=True)

    # Loss plot
    axes[0].plot(epochs, h['loss'], label='Training', linewidth=2)
    axes[0].plot(epochs, h['val_loss'], label='Validation', linewidth=2)
    axes[0].scatter(best_epoch, best_val_loss, color='red', marker='x', s=70, zorder=5, label=f'Best Epoch: {best_epoch}')
    axes[0].set_title(f'{title} - Loss')
    axes[0].set_ylabel('Loss')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    # Accuracy plot
    axes[1].plot(epochs, h['accuracy'], label='Training', linewidth=2)
    axes[1].plot(epochs, h['val_accuracy'], label='Validation', linewidth=2)
    axes[1].scatter(best_epoch, best_val_acc, color='red', marker='x', s=70, zorder=5, label='Accuracy at Best Epoch')
    axes[1].set_title('Accuracy')
    axes[1].set_ylabel('Accuracy')
    axes[1].set_ylim(0, 1.05),
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    # Macro F1 plot
    axes[2].plot(epochs, h['macro_f1'], label='Training', linewidth=2)
    axes[2].plot(epochs, h['val_macro_f1'], label='Validation', linewidth=2)
    axes[2].scatter(best_epoch, best_val_f1, color='red', marker='x', s=70, zorder=5, label='Macro F1 at Best Epoch')
    axes[2].set_title('Macro F1 Score')
    axes[2].set_ylabel('Macro F1')
    axes[2].set_xlabel('Epoch'),
    axes[2].set_ylim(0, 1.05),
    axes[2].legend()
    axes[2].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

def train_and_validate(model,
                       train_data,
                       val_data,
                       epochs               = 500,
                       lr_schedule          = 1e-4,
                       optimizer            = "Adam",
                       title                = "Learning Curves",
                       use_early_stopping   = True,
                       patience             = 5,
                       min_delta            = 1e-4,
                       callbacks            = [],
                       verbose              = 1,
                       return_history       = False
                       ):
    print(f"\n{title}\n")

    # set up optimizer
    if optimizer == "Adam":
        opt = Adam(
                learning_rate   = lr_schedule,
                clipnorm        = 1.0           # prevents exploding gradients
            )
    else:
        opt = optimizer

    # compile model with sparse categorical crossentropy and metrics
    macro_f1 = tf.keras.metrics.F1Score(average='macro', name='macro_f1')
    model.compile(optimizer=opt,
                  loss="categorical_crossentropy",      # labels converted to one-hot encoded tensor
                  metrics=["accuracy", macro_f1]
                )

    # configure early stopping
    early_stopping = EarlyStopping(
        monitor='val_loss',
        patience=patience,
        min_delta=min_delta,
        restore_best_weights=True,   # restores weights from the epoch with minimum validation loss
        verbose=verbose
    )

    if use_early_stopping:
        cbs = [early_stopping] + callbacks
    else:
        cbs = callbacks

    # start timer
    start = time.time()

    # fit model
    history = model.fit(
        train_data,
        epochs          = epochs,
        validation_data = val_data,
        callbacks       = cbs,
        verbose         = verbose
    )

    # identify best epoch (min validation loss)
    best_epoch_idx = np.argmin(history.history['val_loss'])
    best_epoch     = best_epoch_idx + 1
    best_acc       = history.history['val_accuracy'][best_epoch_idx]
    best_f1        = history.history['val_macro_f1'][best_epoch_idx]

    # plot learning curves
    plot_learning_curves(history, title=title)

    print(f"\n--- Training & Validation Summary ---")
    print(f"Best Epoch (Min Val Loss): {best_epoch}")
    print(f"Validation Accuracy at Best Epoch: {best_acc * 100:.2f}%")
    print(f"Validation Macro F1 at Best Epoch: {best_f1 * 100:.2f}%")

    # return execution time
    end = time.time()
    print(f"\nExecution Time: {format_hms(end - start)}")

    # return history
    if return_history:
        return history

def one_hot_encode_dataset(image, label):
    """
        One-hot encodes integer labels for 101 food labels to be compatible with measuring F1 scores.
    """
    label = tf.one_hot(label, depth=101)
    return image, label


# ===================================
# LOAD DATA AND PREPROCESSING STEPS 
# ===================================

# Load Data
food101 = load_dataset("ethz/food101", split="train+validation")

start_time = time.time()        # time preprocessing

# Resize images based on IMAGE_SIZE
resized_food101 = food101.map(
    resize_batch,
    batched=True,
    batch_size=100,
    num_proc=os.cpu_count()          # maximum multi-processing for faster processing
)

# update dataset with cropped images
resized_food101 = resized_food101.map(crop_and_normalize, with_indices=True)

# relabel images first - avoids misaligned indices if I dropped images first
resized_food101 = relabel_image(resized_food101, IMAGES_TO_RELABEL)

# drop images second
resized_food101 = drop_image(resized_food101, IMAGES_TO_DROP)

# drop duplicate images
resized_food101 = drop_image(resized_food101, DUPLICATES_TO_DROP)

# normalize brightness across all images
normalized_brightness_food101 = resized_food101.map(
    normalize_brightness_batch,
    batched=True,
    batch_size=100,
    num_proc=os.cpu_count()          # maximum multi-processing for faster processing
)

# drop dark image and get final cleaned dataset
food101 = drop_image(normalized_brightness_food101, dark_image_idx)


# ====================
# VERIFY CLASS BALANCE 
# ====================

class_counts  = Counter(food101['label'])
label_names   = food101.features['label'].names
class_balance = {label_names[label_id]: count for label_id, count in class_counts.items()}
df = food101.to_pandas()

print('\nClass Counts:', class_balance)
print('Unique Counts per Class:', pd.Series(class_balance.values()).unique())
print('Total images without labels:', df['label'].isna().sum())


# ===========================================
# GET TRAIN/VALIDATION/TEST SPLITS (80/10/10)
# ===========================================

# get 80% train & temp 20% validation/test split
train_test = food101.train_test_split(
    test_size=0.20,
    stratify_by_column="label",
    seed=random_seed
)

# get 10% validation & 10% test splits from the temp 20% split
val_test = train_test["test"].train_test_split(
    test_size=0.50, # 50% of 20% to get 10/10 splits
    stratify_by_column="label",
    seed=random_seed
)

# final train/validation/test splits - stratified to maintain class balances
food101_splits = DatasetDict({
    "train"         : train_test["train"],  # 80%
    "validation"    : val_test["train"],    # 10%
    "test"          : val_test["test"]      # 10%
})

# record duration for preprocessing and splitting data for model training and evaluation
print("\n\nPreprocessing and Data Split Execution Time:", format_hms(time.time() - start_time))

print("\n\n------------     Final Data Splits (80/10/10)       ------------\n")
print(f"                Training Set: {len(food101_splits['train']):,} images.")
print(f"                Validation Set: {len(food101_splits['validation']):,} images.")
print(f"                Test Set: {len(food101_splits['test']):,} images.")


# ====================
# SAVE RESULTS TO DISK
# ====================

# save data splits to disk
target_dir = Path("./cleaned_food101")
temp_dir   = Path("./cleaned_food101_temp")

# save data to temp directory
food101_splits.save_to_disk(str(temp_dir))

# replace old directory with the new one
if target_dir.exists():
    shutil.rmtree(target_dir)
temp_dir.rename(target_dir)

Map (num_proc=18):   0%|          | 0/101000 [00:00<?, ? examples/s]

Map:   0%|          | 0/101000 [00:00<?, ? examples/s]

Map:   0%|          | 0/101000 [00:00<?, ? examples/s]

Relabeled 37 images.


Filter:   0%|          | 0/101000 [00:00<?, ? examples/s]

List of images contained duplicate indices. Dropping 422 images.

New dataset size: 100,578 images.


... (5 outputs left out)

Saving the dataset (0/58 shards):   0%|          | 0/80382 [00:00<?, ? examples/s]

Saving the dataset (0/8 shards):   0%|          | 0/10048 [00:00<?, ? examples/s]

Saving the dataset (0/8 shards):   0%|          | 0/10048 [00:00<?, ? examples/s]

PosixPath('cleaned_food101')

In [5]:
# P1.A — REQUIRED SAVED RESULTS
# Display train/validation/test sample counts and transformed input/label shapes.
# Do not compute test performance here.

# ======================================================================
# RELOAD PREPROCESSED AND SPLIT DATASET (80/10/10 train/validation/test)
# ======================================================================

food101 = load_from_disk("./cleaned_food101")

train   = food101["train"]
val     = food101["validation"]
test    = food101["test"]


# ====================================
# VERIFY SPLIT SHAPE AND SAMPLE COUNTS
# ====================================

# expected image count per split - taken from EDA notebook
expected_img_count = {
    "Train": 80382,
    "Validation": 10048,
    "Test": 10048
}

for split_name, dataset in [("Train", train), ("Validation", val), ("Test", test)]:
    num_images = len(dataset)
    sample_img = np.array(dataset[0]["image"])  # check first image to extract shape later (all images are same size from preprocessing steps, so it's safe to check this way.)
    h, w, c    = sample_img.shape               # height, width, color channels (RGB images: expecting 3 channels)

    if num_images != expected_img_count[split_name]:
        print(f"{split_name} Set doesn't have expected {expected_img_count[split_name]} images: {num_images} images found.")
    else:
        print(f"--- {split_name} Set ---")
        print(f"Total Images: {num_images:,}")
        print(f"Image Dimensions (H x W x Color Channels): {h} x {w} x {c}")
        print(f"Set Shape: ({num_images}, {h}, {w}, {c})\n")

# verify classes are all included in each split
train_labels = food101["train"].features["label"].names
val_labels = food101["validation"].features["label"].names
test_labels = food101["test"].features["label"].names

# ensure there are still 101 labels across all splits
if train_labels == val_labels == test_labels:
    print("\nTrain/Validation/Test Splits all have the same labels")
else:
    print("\nTrain/Validation/Test Splits do not all have the same labels")
    

Loading dataset from disk:   0%|          | 0/58 [00:00<?, ?it/s]

--- Train Set ---
Total Images: 80,382
Image Dimensions (H x W x Color Channels): 512 x 512 x 3
Set Shape: (80382, 512, 512, 3)

--- Validation Set ---
Total Images: 10,048
Image Dimensions (H x W x Color Channels): 512 x 512 x 3
Set Shape: (10048, 512, 512, 3)

--- Test Set ---
Total Images: 10,048
Image Dimensions (H x W x Color Channels): 512 x 512 x 3
Set Shape: (10048, 512, 512, 3)


Train/Validation/Test Splits all have the same labels


**Your Answer:**

> Summarize any preprocessing changes relative to the milestone and why they were necessary. If there are no changes, say so.

*Replace this text with your answer.*

### Part B — Recover the Milestone Baseline (4 pts)

**Evidence (3 pts):** 

- Supply the baseline architecture/configuration and its previously obtained validation results, including the principal selection metric.
- Include training history or relevant plots where applicable.
- Reuse saved milestone results where practical; baseline retraining is not required solely for this section.

**Explanation (1 pt):** State the baseline's primary limitation and the validation metric that will be used to compare later models. Keep metric definitions and validation samples consistent across comparisons.

In [3]:
# P1.B — Bring forward baseline model/configuration, saved history, or summary.
# You may load saved history/results from the milestone or show the relevant code.

# YOUR CODE HERE


In [4]:
# P1.B — REQUIRED SAVED RESULTS
# Show baseline validation metrics and training/validation history or plots.

# YOUR CODE HERE


**Your Answer:**

> State the baseline's primary limitation and the validation metric that will be used to compare later models. Keep metric definitions and validation samples consistent across comparisons.

*Replace this line with your response.*

### Part C — Plan the Final Experiments (2 pts)

- Briefly describe your experimental plan for developing and comparing the custom and pretrained models.
- Identify the improvements or alternatives you intend to investigate, any practical constraints (data subset, training time, compute resources);
    - Include how you will ensure fair comparisons using consistent validation data and metrics.
- Do not use the test set for model selection.

**Your Answer:**

*Replace this text with your answer.*

## Problem 2 — Custom Neural Network and Experiments (35 pts)

- Design and train a more capable, non-pretrained neural network than the milestone baseline, using an architecture appropriate to your dataset.
- Perform at least **two documented runs/configurations** (an initial custom design and at least one intentional improvement experiment).
- The experiments should test a meaningful hypothesis, not merely change a random seed.
- Compare using **validation only**.

> **A. Architecture and implementation (12 pts)**    
> **B. Training and controlled improvement experiment (13 pts)**    
> **C. Results and interpretation (10 pts)**

### Part A — Design and Implement a Custom Model (12 pts)

**Implementation (9 pts):**
- Define and build a custom multiclass neural network trained from scratch (no pretrained weights).
- Present the model architecture, including input representation, principal layers, output layer, regularization, and model summary or equivalent.
- Design an architecture with greater representational capacity or more appropriate inductive biases than the milestone baseline.
- Clearly distinguish model components from preprocessing components.

**Rationale (3 pts):** Explain the principal architectural differences from the milestone baseline, why you chose this design, and how you expect these changes to improve representation learning or generalization.

In [5]:
# P2.A — Define the custom model architecture.
# Include a model summary or equivalent architecture inspection.

# YOUR CODE HERE


In [6]:
# P2.A — REQUIRED SAVED RESULTS
# Display architecture summary and parameter counts when available.

# YOUR CODE HERE


**Your Answer:**

> Explain the principal architectural differences from the milestone baseline, why you chose this design, and how you expect these changes to improve representation learning or generalization.

*Replace this line with your response.*

### Part B — Train and Run a Targeted Improvement Experiment (13 pts)

**Implementation (10 pts):**

- Train the initial custom model and at least one **deliberate variant**.
  - Examples include a justified change to model capacity, regularization, augmentation, optimizer/learning rate, or learning-rate scheduling.
- Document the training configuration for each run, including loss function, optimizer, learning rate, batch size, number of epochs, and any regularization or callbacks used.
- Set reproducible configurations and use a validation-based stopping/selection strategy where appropriate.
- Record training and validation learning curves, the number of epochs actually completed, validation metrics, and approximate training time for each run.
- Use consistent validation data and a clearly stated primary metric to ensure fair comparisons.

**Experiment statement (3 pts):** State the **hypothesis** motivating your improvement experiment, identify what changed between configurations, and explain the expected benefit and any trade-offs. Your hypothesis should be independent of the observed results. **Do not use the test set.**

In [7]:
# P2.B — Train custom initial configuration and at least one deliberate variant.
# Save/history/log configurations and timing. Add more cells as needed.

# YOUR CODE HERE

In [8]:
# P2.B — REQUIRED SAVED RESULTS
# Show a concise run table, validation metrics, learning curves, and runtime.

# YOUR CODE HERE

**Your Answer:**

> State the hypothesis motivating your improvement experiment, identify what changed between configurations, and explain the expected benefit and any trade-offs.

*Replace this text with your answer.*

### Part C — Analyze Custom Model Results (10 pts)

**Results (6 pts):**

- Present an organized, side-by-side **validation comparison** of the milestone baseline, initial custom network, and improved custom network.
  - Include the primary selection metric, at least one supporting metric, and relevant training/validation learning curves.
  - Clearly identify the experimental configurations being compared.
- Identify the best custom configuration using the previously declared selection metric. **Do not use test results for model selection.**

**Interpretation (4 pts):**

- Evaluate whether your improvement hypothesis was supported by the validation results, citing specific evidence.
- Discuss evidence of underfitting or overfitting, generalization, and any trade-offs between model performance and computational cost.
- Explain what you learned about model architecture, training, or optimization and how these findings will inform your subsequent experiments.
- If the custom models did not outperform the milestone baseline, explain what the results suggest rather than concealing or dismissing the outcome.

In [9]:
# P2.C — Compare validation results.
# Create a table comparing the milestone baseline,
# initial custom network, and improved custom network.
# Include configurations and validation metrics.

# YOUR CODE HERE


In [10]:
# P2.C — Visualize training and validation behavior.
# Plot clearly labeled learning curves for the custom models.
# Include any additional plots useful for interpreting
# generalization, underfitting, or overfitting.

# YOUR CODE HERE


**Interpretation: Your Answer:**

*Replace this text with your answer.*

## Problem 3 — Pretrained Models and Transfer Learning (35 pts)

In this problem, you will investigate whether **transfer learning** can improve performance on your classification task. Select an appropriate pretrained model, adapt it to your dataset, and evaluate different strategies for using its pretrained representations.

You will:

- Select a suitable pretrained architecture and adapt its classification head to your task.
- Implement and train an initial **frozen-backbone (feature-extraction)** model.
- Experiment with at least one additional transfer-learning strategy, such as partial unfreezing, fine-tuning, or parameter-efficient adaptation.
- Compare the configurations using **validation results only**, considering both predictive performance and computational efficiency.

> **A. Pretrained model selection and implementation (12 pts)**  
> **B. Transfer-learning and fine-tuning experiment (13 pts)**  
> **C. Results, efficiency, and interpretation (10 pts)**

### Part A —Select and Implement a Pretrained Model (12 pts)

**Implementation (9 pts):**

- Select a publicly available pretrained architecture and identify its pretrained weights or checkpoint.
- Apply the required preprocessing, such as image transformations or text tokenization.
- Adapt the classification head to the number of classes in your dataset.
- Configure an initial **frozen-backbone model**, keeping the pretrained backbone fixed and the classification head trainable. Training is performed in Part B.
- Display the model architecture or summary, including total and trainable parameter counts.

**Rationale (3 pts):**

Explain why the pretrained model is appropriate for your classification task. Discuss the potential benefits of its learned representations and any relevant limitations, such as domain mismatch, model complexity, or computational requirements.

In [11]:
# P3.A — Load the pretrained model and apply the required preprocessing.
# Adapt the classification head and freeze the pretrained backbone,
# leaving the classification head trainable.

# YOUR CODE HERE


In [12]:
# P3.A — Display the pretrained model configuration.
# Show the model architecture, pretrained checkpoint, classification head,
# and total, trainable, and frozen parameter counts.

# YOUR CODE HERE


**Your Answer:**

> Explain why the pretrained model is appropriate for your classification task. Discuss the potential benefits of its learned representations and any relevant limitations, such as domain mismatch, model complexity, or computational requirements.

*Replace this text with your answer.*

### Part B — Train and Evaluate Transfer-Learning Strategies (13 pts)

**Implementation (10 pts):** Train a frozen-feature model and at least one controlled adaptation strategy, such as partial unfreezing, full fine-tuning, or parameter-efficient fine-tuning. Document the learning rate(s), optimizer, epochs, stopping criterion, trainable layers, batch size, and training time. Use **validation** for comparison; do not use the test set. If full fine-tuning is infeasible, choose and justify a feasible alternative rather than claiming it was performed.

**Experiment statement (3 pts):** State the rationale and hypothesis for the adaptation strategy and explain how you guarded against unstable updates or overfitting.

In [13]:
# P3.B — Train the frozen-backbone model from Part A.
# Implement and train at least one additional adaptation strategy
# (e.g., partial unfreezing, full fine-tuning, or parameter-efficient tuning).
# Record training configurations, histories, and training times.

# YOUR CODE HERE


In [14]:
# P3.B — Summarize results for each transfer-learning configuration.
# Display validation metrics, training/validation learning curves,
# epochs completed, and approximate training times.
# Do not evaluate on the test set.

# YOUR CODE HERE


**Your Answer:**

> State the rationale and hypothesis for the adaptation strategy and explain how you guarded against unstable updates or overfitting.

*Replace this text with your answer.*

### Part C — Compare Performance and Computational Efficiency (10 pts)

**Results (6 pts):** 

- Compare the pretrained configurations with each other and with the best custom model on the **same or explicitly comparable validation data**.
- Include the primary metric, secondary metric(s), approximate runtime, and computational cost/proxy (e.g., trainable parameters, batch size, or memory if measured).
- Identify the preferred pretrained configuration using the previously declared primary validation metric.

**Interpretation (4 pts):** 

- Discuss gains, failures, convergence, generalization, and the trade-offs associated with pretraining and fine-tuning.
- State explicitly if comparisons are limited by differences in datasets, training budgets, or computational resources.
- Discuss whether the performance improvements justify the additional computational cost.

In [15]:
# P3.C — Compare validation results for the pretrained configurations
# and the best custom model from Problem 2.
# Display primary and supporting metrics, training times,
# and relevant measures of computational cost.
# Use the same or explicitly comparable validation data.

# YOUR CODE HERE


In [16]:
# P3.C — Visualize performance and computational trade-offs.
# Create plots comparing validation performance with training time
# or another meaningful measure of computational cost.
# Clearly label models and configurations.

# YOUR CODE HERE


**Interpretation: Your Answer:**


*Replace this text with your answer.*

## Problem 4 — Model Comparison, Selection, and Final Training (30 pts)

Make a principled final choice among the baseline, best custom model, and best pretrained model. The decision must be based **only on validation results and legitimate resource constraints**. Then finalize the selected model, document its training configuration, and establish the protocol for final test-set evaluation.

- **A. Integrated validation comparison (12 pts)**
- **B. Selection and final configuration (10 pts)**
- **C. Final training and evaluation readiness (8 pts)**

### Part A —Compare the Three Model Families (12 pts)

**Results (8 pts):**

- Present a side-by-side **validation comparison** of the milestone baseline, best custom model, and best pretrained model.
- Include the primary selection metric, supporting metrics, and relevant measures of computational cost or model complexity.
- Examine per-class validation performance, using tables or plots where appropriate to identify differences between models.
- Use the same or explicitly comparable validation data. If datasets, subsets, or evaluation procedures differ, explain the limitations and make comparisons consistent where feasible.

**Interpretation (4 pts):**

- Discuss the relative strengths and weaknesses of the three model families, including class-specific performance and evidence of overfitting or underfitting.
- Evaluate trade-offs between predictive performance, model complexity, and computational cost.
- Recommend a preferred model based on the validation evidence, explaining why it represents the best overall choice rather than relying on a single metric.

In [17]:
# P4.A — Consolidate validation results from Problems 1–3.
# Assemble the metrics and computational cost measures
# for the milestone baseline, best custom model, and best pretrained model.

# YOUR CODE HERE


In [18]:
# P4.A — Display the final three-model validation comparison.
# Show primary and supporting metrics, computational costs,
# and relevant per-class diagnostic tables or plots.

# YOUR CODE HERE


**Interpretation: Your Answer:**

*Replace this text with your answer.*

### Part B — Choose and Lock the Final Model (10 pts)

**Choice and Rationale (6 pts):**

- Identify your selected model family and configuration.
- Justify your selection based on validation performance, the primary selection metric, and computational constraints.
- Briefly explain why the principal alternatives were not selected.
- State the exact primary selection metric and any tie-breaking criteria.

**Locked Configuration (4 pts):**

Complete the provided `final_config` dictionary, replacing the example values with your final choices. Include:

- Model architecture and pretrained checkpoint (if applicable).
- Preprocessing and training data/subset policy.
- Training hyperparameters, including optimizer, learning rate, and batch size.
- Random seed, stopping criteria, and checkpoint/epoch selection rule.
- Primary and supporting metrics for final evaluation.

**All modeling and evaluation decisions must be fixed before examining test results in Problem 5.**

In [19]:
# P4.B — Document the final model configuration.
# Replace the example values below with your selected model and settings.
# Add or modify entries as appropriate for your architecture.
# All decisions must be fixed before evaluating on the test set.

final_config = {
    "model_family": "pretrained",
    "architecture": "ResNet50",
    "pretrained_checkpoint": "ImageNet weights",
    "preprocessing": "Resize to 224x224, normalize",
    "training_data": "Training split only (or combined train+validation for fixed-epoch retraining)",
    "validation_policy": "Original validation split",
    "trainable_layers": "Classification head and final backbone block",
    "optimizer": "Adam",
    "learning_rate": 1e-5,
    "batch_size": 32,
    "max_epochs": 30,
    "stopping_rule": "Early stopping, patience=5",
    "checkpoint_rule": "Best validation macro F1",
    "random_seed": 42,
    "primary_metric": "Macro F1",
    "supporting_metrics": ["Accuracy", "Weighted F1"],
    "tie_breaker": "Lower computational cost",
}

# Display the locked configuration.
final_config

{'model_family': 'pretrained',
 'architecture': 'ResNet50',
 'pretrained_checkpoint': 'ImageNet weights',
 'preprocessing': 'Resize to 224x224, normalize',
 'training_data': 'Training split only (or combined train+validation for fixed-epoch retraining)',
 'validation_policy': 'Original validation split',
 'trainable_layers': 'Classification head and final backbone block',
 'optimizer': 'Adam',
 'learning_rate': 1e-05,
 'batch_size': 32,
 'max_epochs': 30,
 'stopping_rule': 'Early stopping, patience=5',
 'checkpoint_rule': 'Best validation macro F1',
 'random_seed': 42,
 'primary_metric': 'Macro F1',
 'supporting_metrics': ['Accuracy', 'Weighted F1'],
 'tie_breaker': 'Lower computational cost'}

**Choice and Rationale: Your Answer:**

*Explain why you selected this final model based on validation results and resource constraints, identify your primary selection metric and any tie-breaker, and briefly justify rejecting the principal alternatives.*

*Replace this text with your answer.*

### Part C — Produce the Final Trained Model (8 pts)

**Implementation and Evidence (6 pts):**

- Train or restore the selected model using the locked configuration from Part B.
- Choose one of the following approaches:
  - **Retain a validation partition:** Train the model using the training data, with validation-based early stopping or checkpoint selection.
  - **Retrain on combined data:** Use the results of previous validation experiments to establish a fixed training duration, then retrain using the combined training and validation data.
  - **Restore a checkpoint:** Load an already-trained model selected using validation results.
- Record the actual training procedure, including the data used, epochs completed, and final checkpoint or model.
- Save the trained model or checkpoint for reuse, where practical.

**Do not use the test set for training, early stopping, checkpoint selection, or any other model-selection decisions.**

**Protocol Explanation (2 pts):**

- Explain which data were used for final training and validation, if applicable.
- State whether the final model was retrained or restored from a previously selected checkpoint.
- Explain how the final training procedure follows the configuration established in Part B.

In [20]:
# P4.C — Train or restore the locked final model from Part B.
# Follow the declared training protocol without using test data.
# Save the final model or checkpoint where practical.

# YOUR CODE HERE


In [21]:
# P4.C — Display evidence of the completed final model.
# Show whether the model was trained or restored, the data used,
# epochs completed or checkpoint restored, and random seed.
# Identify the saved model/checkpoint, if applicable.
# Do not include test results.

# YOUR CODE HERE


**Protocol Explanation: Your Answer:**

*Replace this text with your answer.*

## Problem 5 — Final Test Evaluation and Error Analysis (25 pts)

In this problem, you will evaluate your **locked final model** from Problem 4 on the held-out test set and analyze its performance.

- Evaluate the final model on the test set **only once**, using the previously established evaluation protocol.
- Report the results accurately, including any unexpected or disappointing outcomes.
- Do not modify the model, hyperparameters, or evaluation procedure in response to test results.

**This is the first point in the assignment where test-set performance should be examined or reported.**

> **A. Final test metrics and visualizations (10 pts)**  
> **B. Error analysis (9 pts)**  
> **C. Interpretation and limitations (6 pts)**

### Part A —Evaluate the Locked Model on Test Data (10 pts)

**Implementation and Results (8 pts):**

- Apply the appropriate inference preprocessing and evaluate the locked final model on the held-out test set.
- Report test **accuracy**, **macro-F1**, and any additional metrics used in your validation comparisons.
- Present a labeled confusion matrix (or a readable summary for datasets with many classes).
- Include per-class performance metrics, such as precision, recall, and F1, to identify differences across classes.

**Methodology (2 pts):**

- Report the test-set size, number of classes, and final model/checkpoint used.
- Explain how the test set was kept separate from training, validation, and model selection.
- State whether the evaluation used the entire test set or a subset, and explain any restrictions.

In [22]:
# P5.A — FIRST TEST-SET EVALUATION.
# Evaluate ONLY the locked final model from Problem 4.
# Apply the required inference preprocessing and generate test predictions.
# Compute accuracy, macro-F1, supporting metrics, and confusion matrix.
# Do not modify or retrain the model based on test results.

# YOUR CODE HERE


In [23]:
# P5.A — Display final test-set evaluation results.
# Show accuracy, macro-F1, supporting metrics, and per-class performance.
# Display a labeled confusion matrix (or readable summary for many classes).
# Report the test-set size and number of classes.

# YOUR CODE HERE


**Methodology: Your Answer:**

*Replace this text with your answer.*

### Part B — Analyze Errors (9 pts)

**Evidence (6 pts):** Identify the hardest classes and at least two recurring types of errors, using **test-set** confusion patterns, per-class metrics, or representative misclassified examples. When showing examples, use only data appropriate to share and label model predictions and true labels clearly. For a large number of classes, summarize or focus the visualization rather than presenting an unreadable full matrix.

**Interpretation (3 pts):** Explain plausible mechanisms behind those errors, distinguishing evidence from speculation.

In [24]:
# P5.B — Analyze test-set classification errors.
# Identify the hardest classes using per-class metrics and confusion patterns.
# Find at least two recurring types of misclassification.
# Select representative examples where appropriate.

# YOUR CODE HERE


In [25]:
# P5.B — Display error-analysis results.
# Show the hardest classes and their relevant performance metrics.
# Display evidence of at least two recurring error patterns,
# using confusion summaries or labeled misclassified examples.
# Avoid unreadable plots for datasets with many classes.

# YOUR CODE HERE


**Your Answer:**

> Explain plausible mechanisms behind those errors, distinguishing evidence from speculation.

*Replace this text with your answer.*

### Part C — Interpret the Final Outcome (6 pts)

**Interpretation and Generalization:**

- Compare the final test performance with the validation results. Discuss whether the results suggest that the model generalizes well to unseen data.
- Explain any notable differences between validation and test performance.

**Limitations and Future Improvements:**

- Identify **two specific limitations** of the final model or experimental approach, supported by your results.
- Propose **two concrete improvements** that could address these limitations, explaining why they might help.

**Final Reflection:**

- Summarize the most important technical lesson you learned about model development, training, or evaluation.

**Do not modify or reevaluate the model based on test results.** Proposed improvements are for future work. The technical report will explore these findings in greater detail.

**Your Answer:**

*Replace this text with your answer.*

## AI-Use Disclosure (required; no separate points)

Consistent with CDS policy, disclose any generative AI tools used in the project. State what tasks the tools assisted with, what you independently verified, and important limitations or errors you found. If you did not use such tools, say so.

**Your Answer:**

*Replace this text with your answer.*